In [6]:
import logging
logging.basicConfig(
	filename='app.log',
	level=logging.INFO,
	filemode='w',  # 'w' for write (overwrite), 'a' for append (default)
	format='%(asctime)s - %(levelname)s - %(message)s'
)
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from core.Log import *
from core.preprocessing import *
from core.plots import *
from core.mydataloader import *

CNTRL_dicom_root = "../Takotsubo-Syndrome/data/Inputs/normal_cases/"
TTS_dicom_root = "../Takotsubo-Syndrome/data/Inputs/takotsubo_cases/"
root_dir = "data/cases/"



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:

def run_one_fold(train_datalist, val_datalist, test_datalist, fold_idx):
	"""
	Executes the entire pipeline for a single fold of the cross-validation.
	This includes:
	1. Calculating normalization stats for the fold's training data.
	2. Creating DataLoaders.
	3. Initializing and training the model.
	4. Evaluating the model on the fold's test set.
	Args:
		train_datalist (list): The list of training cases for this fold.
		val_datalist (list): The list of validation cases for this fold.
		test_datalist (list): The list of test cases for this fold.
		fold_idx (int): The index of the current fold (for logging).
	Returns:
		float: The performance score (e.g., AUC) for this fold.
	"""
	logging.info(f"--- Starting Fold {fold_idx + 1} ---")
	logging.info(f"Fold Split: {len(train_datalist)} train, {len(val_datalist)} val, {len(test_datalist)} test.")


	HUstats = [case["stats"] for case in train_datalist]
	UH_mean, HU_std = calculate_HU_stats(HUstats)

	ages = [case['age'] for case in train_datalist]
	AGE_mean = np.mean(ages); AGE_std = np.std(ages)
	logging.info(f"Fold {fold_idx + 1} | HU_mean={UH_mean:.2f}, HU_std={HU_std:.2f}, AGE_mean={AGE_mean:.2f}, AGE_std={AGE_std:.2f}")
	fold_stats = {
		'HU_mean': UH_mean,
		'HU_std': HU_std,
		'AGE_mean': AGE_mean,
		'AGE_std': AGE_std}

	train_loader, val_loader, test_loader = get_data_loaders(
	     				train_datalist,
						val_datalist,
						test_datalist,
						fold_stats)

	# 3. Initialize model and train it
	# This function would contain your epoch loop, training, validation, and saving the best model
	# trained_model = train_and_evaluate_model(train_loader, val_loader)

	# 4. Evaluate the final model on the held-out test set
	# score = evaluate_final_model(trained_model, test_loader)
	score = np.random.rand() # Placeholder for the actual score
	logging.info(f"--- Fold {fold_idx + 1} Score: {score:.4f} ---")

	return score




In [ ]:
def main_training_pipeline():

	# --- 1. Load the datalist ---
	json_path = Path('data/data_info.json')
	full_datalist = load_dataset_info(json_path)
	if not full_datalist:
		logging.error("Datalist not loaded.")
		return

	# Prepare indices and labels for stratified splitting
	indices = np.arange(len(full_datalist))
	labels = [d['label'] for d in full_datalist]

	# --- 2. Setup the Outer Cross-Validation Loop ---
	N_OUTER_SPLITS = 5
	outer_cv = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=42)

	fold_scores = []

	logging.info(f"Starting {N_OUTER_SPLITS}-fold Nested Cross-Validation...")

	# The loop that creates the 5 different test sets
	for fold_idx, (train_val_idx, test_idx) in enumerate(outer_cv.split(indices, labels)):

		# --- 3. Outer Split: (Train+Val Pool) vs. Test Set ---
		# This creates the held-out test set for the current fold
		test_datalist_fold = [full_datalist[i] for i in test_idx]

		# This creates the pool of data that will be used for training and validation
		train_val_datalist_fold = [full_datalist[i] for i in train_val_idx]
		train_val_labels_fold = [d['label'] for d in train_val_datalist_fold]

		# --- 4. Inner Split: Train Set vs. Validation Set ---
		# Now we split the pool from the step above.
		# test_size=0.25 on the remaining 80% gives an overall 60% train / 20% val / 20% test split.
		inner_sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
		train_idx, val_idx = next(inner_sss.split(np.arange(len(train_val_datalist_fold)), train_val_labels_fold))

		# Create the final datalists for this fold
		train_datalist_fold = [train_val_datalist_fold[i] for i in train_idx]
		val_datalist_fold = [train_val_datalist_fold[i] for i in val_idx]

		# --- 5. Run the Training and Evaluation for this Fold ---
		# Pass the three distinct datasets to your core function
		score = run_one_fold(
			train_datalist=train_datalist_fold,
			val_datalist=val_datalist_fold,
			test_datalist=test_datalist_fold,
			fold_idx=fold_idx
		)
		fold_scores.append(score)

	# --- 6. Report Final Results ---
	mean_score = np.mean(fold_scores)
	std_score = np.std(fold_scores)

	logging.info("--- Nested Cross-Validation Complete ---")
	logging.info(f"Final Scores across {N_OUTER_SPLITS} folds: {[f'{s:.4f}' for s in fold_scores]}")
	logging.info(f"Average Model Performance: {mean_score:.4f} ± {std_score:.4f}")


Loading cropped images: 100%|██████████| 157/157 [00:28<00:00,  5.59it/s]


In [ ]:
main_training_pipeline()
